In [ ]:
import os
import nibabel as nib
import pandas as pd

# === Configuration des groupes
group_info = {
    'Groupe1': 'Tollsome_output1',
    'Groupe2': 'Tollsome_output2',
    'Groupe3': 'Tollsome_output3'
}

# === Répertoires de base
base_dir = '/home/amenacer/Stage/Data/Segmentation'
images_base = os.path.join(base_dir, 'Tollsome')
masks_base = os.path.join(base_dir, 'Tollsomemask')
outputs_base = os.path.join(base_dir, 'resultas')

# === Liste des données
recap_data = []

# === Parcours des groupes
for group, output_folder in group_info.items():
    images_dir = os.path.join(images_base, group)
    masks_dir = os.path.join(masks_base, group)
    output_dir = os.path.join(outputs_base, output_folder)

    for filename in os.listdir(images_dir):
        if filename.endswith('.nii.gz'):
            image_path = os.path.join(images_dir, filename)
            mask_path = os.path.join(masks_dir, filename.replace('_0000', ''))
            output_path = os.path.join(output_dir, filename)

            entry = {
                'Groupe': group,
                'Fichier': filename,
                'Dimensions': '',
                'Masque trouvé': os.path.exists(mask_path),
                'Résultat généré': os.path.exists(output_path)
            }

            try:
                img_data = nib.load(image_path).get_fdata()
                # Squeeze si image 4D avec dernier axe=1
                if img_data.ndim == 4 and img_data.shape[-1] == 1:
                    img_data = img_data.squeeze(-1)
                entry['Dimensions'] = str(img_data.shape)
            except Exception as e:
                entry['Dimensions'] = 'Erreur'
                entry['Remarque'] = str(e)

            recap_data.append(entry)

# === Création du CSV
df = pd.DataFrame(recap_data)
csv_path = os.path.join(outputs_base, 'recap_global.csv')
df.to_csv(csv_path, index=False)
print(f"📄 CSV global généré : {csv_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# === Lecture du CSV global
csv_path = '/home/amenacer/Stage/Data/Segmentation/resultas/recap_global.csv'
df = pd.read_csv(csv_path)

# === Dossier pour sauvegarder les graphiques
output_plot_dir = '/home/amenacer/Stage/Data/Segmentation/resultas/plots'
os.makedirs(output_plot_dir, exist_ok=True)

# === Nettoyage des valeurs manquantes
df.fillna('', inplace=True)

# === STATISTIQUES GÉNÉRALES
total_images = len(df)
total_groupes = df['Groupe'].nunique()
traitées = df[df['Résultat généré'] == True].shape[0]
sans_masque = df[df['Masque trouvé'] == False].shape[0]

print(f"Total d'images       : {total_images}")
print(f"Nombre de groupes    : {total_groupes}")
print(f"Images traitées      : {traitées}")
print(f"Images sans masque   : {sans_masque}")

# === GRAPHIQUE 1 : Images traitées par groupe
df.groupby('Groupe')['Résultat généré'].sum().plot(kind='bar', color='mediumseagreen')
plt.title("Images traitées par groupe")
plt.ylabel("Nombre d'images")
plt.xlabel("Groupe")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_plot_dir, 'images_traitees_par_groupe.png'))
plt.close()

# === GRAPHIQUE 2 : Masques manquants par groupe (si applicable)
masques_manquants = df[df['Masque trouvé'] == False]['Groupe'].value_counts()

if not masques_manquants.empty:
    masques_manquants.plot(kind='bar', color='tomato')
    plt.title("Masques manquants par groupe")
    plt.ylabel("Nombre de fichiers sans masque")
    plt.xlabel("Groupe")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(os.path.join(output_plot_dir, 'masques_manquants_par_groupe.png'))
    plt.close()
else:
    print("Tous les fichiers ont un masque. Aucun graphique d'erreur généré.")

# === GRAPHIQUE 3 : Nombre total de fichiers par groupe
df['Groupe'].value_counts().plot(kind='bar', color='cornflowerblue')
plt.title("Nombre total de fichiers par groupe")
plt.ylabel("Nombre d'images")
plt.xlabel("Groupe")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(output_plot_dir, 'fichiers_totaux_par_groupe.png'))
plt.close()

print(f"Graphiques enregistrés dans : {output_plot_dir}")
